# 📊 01: Data Exploration
**GUARDIAN-NLP** | United Nations AI Safety Division

This notebook explores the raw and external datasets before preprocessing.

## 1. Load External Datasets

In [1]:
# Load Jigsaw dataset
jigsaw_path = '../data/external/jigsaw_toxic.csv'
if os.path.exists(jigsaw_path):
    jigsaw = pd.read_csv(jigsaw_path)
    print(f'Jigsaw: {len(jigsaw):,} rows, columns: {jigsaw.columns.tolist()}')
    display(jigsaw.head(3))
else:
    print('Jigsaw not found. Download from https://www.kaggle.com/c/jigsaw-toxic-comment-classification-challenge')

NameError: name 'os' is not defined

In [ ]:
# Load Davidson dataset
davidson_path = '../data/external/hate_speech_davidson.csv'
if os.path.exists(davidson_path):
    davidson = pd.read_csv(davidson_path)
    print(f'Davidson: {len(davidson):,} rows')
    print(davidson['class'].value_counts())
else:
    print('Davidson not found.')

In [ ]:
# Load Depression Reddit from HuggingFace
try:
    from datasets import load_dataset
    depression_ds = load_dataset('mrjunos/depression-reddit-cleaned')
    depression_df = depression_ds['train'].to_pandas()
    print(f'Depression Reddit: {len(depression_df):,} rows')
    print(depression_df['label'].value_counts())
except Exception as e:
    print(f'Could not load: {e}')

## 2. Label Distribution Analysis

In [ ]:
# Load combined dataset if available
combined_path = '../data/raw/combined_raw.csv'
if os.path.exists(combined_path):
    df = pd.read_csv(combined_path)
    print(f'Combined dataset: {len(df):,} rows')
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Label distribution
    df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['#2ecc71','#f39c12','#e74c3c','#8e0000'])
    axes[0].set_title('Label Distribution (Combined)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Label')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=0)
    
    # Source dataset distribution
    df['source_dataset'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
    axes[1].set_title('Data Sources', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.savefig('../outputs/visualizations/label_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Chart saved to outputs/visualizations/label_distribution.png')
else:
    print('Run data_merger.py first to create combined_raw.csv')

## 3. Text Length Analysis

In [ ]:
if os.path.exists(combined_path):
    df['text_length'] = df['text'].astype(str).apply(lambda x: len(x.split()))
    
    print('Text length statistics:')
    print(df.groupby('label')['text_length'].describe().round(1))
    
    fig, ax = plt.subplots(figsize=(12, 5))
    for label in df['label'].unique():
        subset = df[df['label'] == label]['text_length']
        subset.plot(kind='hist', alpha=0.5, bins=50, ax=ax, label=label)
    ax.set_xlabel('Number of Words')
    ax.set_title('Text Length Distribution by Label')
    ax.axvline(x=128, color='red', linestyle='--', label='BERT max (128)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    print(f'Posts > 128 words: {(df.text_length > 128).mean():.1%} (will be truncated)')

## 4. Sample Texts per Category

In [ ]:
if os.path.exists(combined_path):
    for label in ['normal', 'depressive', 'hate_speech', 'violent']:
        subset = df[df['label'] == label]
        if not subset.empty:
            print(f'\n[{label.upper()}] (n={len(subset):,})')
            for txt in subset['text'].sample(min(3, len(subset)), random_state=42):
                print(f'  → {str(txt)[:120]}...')